# Õppetund 10 - Tehisintellekti agendid tootmiskeskkonnas

Selles õppetükis õpid AI agentide **tootmismustreid** Microsoft Agent Frameworki (Python) abil. Käsitleme:

- **Jälgitavus** — agentide suhtluste ajastamise ja logimise lisamine
- **Hindamine** — hindava agendi kasutamine vastuste kvaliteedi skoorimiseks
- **Kulukontroll** — strateegiad tokenite optimeerimiseks ja mudelite valimiseks

Stsenaariumiks on **reisibüroo**, mis aitab kasutajatel reisiplaane teha, millele on lisatud jälgimine ja hindamine.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tootmise kaalutlused

AI-agentide liigutamine prototüüpidest tootmisse nõuab hoolikat tähelepanu kolmele sambale:

1. **Jälgitavus** — Sul peab olema ülevaade sellest, mida agent teeb, kui kaua see aega võtab ja milliseid tööriistu ta kutsub. Ilma jälgimise ja logimiseta on tootmisvigade silumine praktiliselt võimatu.

2. **Hindamine** — Automaatne kvaliteedikontroll tagab, et agendi vastused jäävad aja jooksul täpsed, täielikud ja kasulikud. Hindajaagent võib vastuseid defineeritud kriteeriumite alusel skoorida.

3. **Kulu juhtimine** — Sümbolite kasutus mõjutab kulusid otseselt. Strateegiad nagu üleskutsete optimeerimine, mudeli valik ja vahemällu salvestamine aitavad hoida kulud kontrolli all ilma kvaliteedist loobumata.


## Objektiivse agendi loomine

Me määratleme reisivahendid ja mähkime agendi kutsumise ajastusega, et saaksime jälgida latentsust. Tootmiskeskkonnas integreeriksite OpenTelemetry või mõne muu sarnase jälgimissüsteemiga.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Hindamismustrid

Tavapärane tootmisviis on kasutada teist agenti kui **hinnatajat**. Hinnataja hindab peamise agendi vastust eelnevalt määratletud kriteeriumide alusel nagu täielikkus, täpsus ja kasulikkus.

See võimaldab:
- Automatiseeritud kvaliteedikontrolli enne vastuste kasutajatele jõudmist
- Tagasikerimise avastamist, kui juhiseid või mudeleid muudetakse
- Agendi jõudluse pidevat jälgimist aja jooksul


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kulude haldamise strateegiad

Kulude kontrollimine on tootmises AI agentide jaoks kriitilise tähtsusega. Siin on peamised strateegiad:

| Strateegia | Kirjeldus |
|---|---|
| **Prompti optimeerimine** | Hoidke süsteemi juhised lühikesed. Eemaldage liigset konteksti, et vähendada sisendtokenite arvu. |
| **Mudeli valik** | Kasutage väiksemaid, odavamaid mudeleid (nt GPT-4o-mini) lihtsate ülesannete jaoks nagu klassifitseerimine või ekstreem, ning hoidke suuremaid mudeleid keerulisema mõtlemise jaoks. |
| **Vahemällu salvestamine** | Salvestage tööriistade tulemused ja sagedased päringud, et vältida liigseid API üleskutseid. |
| **Tokeni eelarved** | Määrake `max_tokens` piirid, et vältida ootamatult pikki vastuseid. |
| **Grupeerimine** | Ühendage mitu kasutajapäringut võimalusel üheks API-kõneks. |

Praktiliselt töötab hästi kihiline lähenemine: suunake lihtsad päringud kiirele, odavale mudelile ja keerulised päringud eskaleerige võimekamale mudelile.


## Kokkuvõte

Selles õppetükis õppisid, kuidas:

1. **Lisada vaatlusvõimekus** agendi suhtlustes ajastuse ja logimise abil, mis loob aluse jälgimiseks ja monitooringuks.
2. **Hinnata automaatselt agendi vastuseid** hindamisagendi abil, mis annab skoori täielikkuse, täpsuse ja abistavuse põhjal.
3. **Haldada kulusid** käskluste optimeerimise, mudeli valiku, vahemällu salvestamise ja tokeni eelarvete kaudu.

Need tootmismustrid aitavad tagada, et teie tehisintellekti agendid on usaldusväärsed, mõõdetavad ja kulutõhusad suures mahus.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
